In [ ]:
import os
import pandas as pd
import numpy as np
import PIL.Image as Image
from torchvision import models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torchmetrics.classification import MulticlassF1Score
from torchmetrics import Accuracy
import torch.nn.init as init
import torch.optim as optim
from torch import nn
import torch
import os
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from matplotlib import pyplot as plt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'{device} is available as our device')

In [ ]:
class PlantDiseaseModel(nn.Module):
    def __init__(self, num_plant_classes: int, num_disease_classes :int):
        super(PlantDiseaseModel, self).__init__()
        base_model = models.mobilenet_v3_small(weights='DEFAULT')
        self.shared = nn.Sequential(
            base_model.features,
            base_model.avgpool
        )

        in_features = 576
        self.disease_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features, 1024),

            nn.Hardswish(),
            nn.Dropout(0.3, inplace=True),
            nn.Linear(1024, num_disease_classes)
        )

        self.plant_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features, 1024),

            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, num_plant_classes)
        )
    
    def forward(self, x):

        features = self.shared(x)
        plant_outputs = self.plant_head(features)
        disease_outputs = self.disease_head(features)

        return plant_outputs, disease_outputs

Train mean = [0.46160856 0.55330724 0.3025134 ]
Train std  = [0.16686249 0.17260724 0.15989958]

In [ ]:
transform = transforms.Compose([

    transforms.RandomAdjustSharpness(1.5),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),

    transforms.RandomResizedCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.4616, 0.5533, 0.3025), (0.1669, 0.1726, 0.1599))
    
    ])

In [ ]:
class CropDiseaseDataset(Dataset):
    def __init__(self, csv_file: str, transform=None):
        super().__init__()
        df = pd.read_csv(csv_file)
        self.data = df.to_numpy()

        self.transform = transform
        self.disease_to_idx = {name: i for i, name in enumerate(np.unique(self.data[:, 1]))}
        self.plant_to_idx = {name: i for i, name in enumerate(np.unique(self.data[:, 0]))}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index: int):
        image_path = self.data[index, 2]
        image = Image.open(image_path).convert('RGB')

        crop_type = self.plant_to_idx[self.data[index, 0]]
        disease_type = self.disease_to_idx[self.data[index, 1]]
        
        if image:
            image = self.transform(image)
        return image, torch.tensor(crop_type), torch.tensor(disease_type)

In [ ]:
class PredictPlantDisease:
    def __init__(self, image_path: str, model_path : str,  plant_classes: list, disease_classes: list):
        self.model = self._load_model(model_path)
        self.plant_classes = plant_classes
        self.disease_classes = disease_classes
        self.image_path = image_path

        self.transforms = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize((0.4616, 0.5533, 0.3025), (0.1669, 0.1726, 0.1599))
        ])
    
    def _load_model(self, model_path):
        model = PlantDiseaseModel(6, 27)
        model.load_state_dict(torch.load(model_path))
        model.eval()
        return model
    
    def _preprocess(self, image_path):
        image = Image.open(image_path).convert('RGB')
        image = self.transforms(image).unsqueeze(0)
        return image
    
    def _format_results(self, p_probs, d_probs):
        plant_idx = torch.argmax(p_probs)
        plant_confidence = p_probs[0][plant_idx].item()
        plant_name = self.plant_classes[plant_idx]

        if plant_name == 'Unknown' and plant_confidence >= 0.7:
            return {'status':'error', 'message':'Unknown object detected'}
        
        disease_idx = torch.argmax(d_probs)
        disease_confidence = d_probs[0][disease_idx].item()
        disease_name = self.disease_classes[disease_idx]

        return {
            'status':'success', 'plant':plant_name, 'disease':disease_name,
            'confidence':{'plant':plant_confidence, 'disease':disease_confidence}
        }
    def run_model(self):
        image_tensor = self._preprocess(image_path=self.image_path)

        with torch.no_grad():
            plant_outputs, disease_outputs = self.model(image_tensor)
            plant_probs = torch.softmax(plant_outputs, dim=1)
            disease_probs = torch.softmax(disease_outputs, dim=1)
        
        return self._format_results(plant_probs, disease_probs)

In [ ]:
BASE_PATH = os.getcwd()[:-9]

train_path = os.path.join(BASE_PATH, 'data/train.csv')
test_path = os.path.join(BASE_PATH, 'data/test.csv')
val_path = os.path.join(BASE_PATH, 'data/val.csv')

train_data = CropDiseaseDataset(train_path, transform=transform)
test_data = CropDiseaseDataset(test_path, transform=transform)
val_data = CropDiseaseDataset(val_path, transform=transform)

train_loader = DataLoader(train_data, batch_size=30, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=32)
val_loader = DataLoader(val_data, batch_size=32)

In [ ]:
model = PlantDiseaseModel(5, 17)
model = model.to(device)
model.to(device)

In [ ]:
EPOCHS = 80

optimizer = optim.AdamW(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

f1score_plant = MulticlassF1Score(num_classes=5, average="macro").to(device)
f1score_disease = MulticlassF1Score(num_classes=26, average="macro").to(device)

accuracy_plant = Accuracy(task='multiclass', num_classes=6).to(device)
accuracy_disease = Accuracy(task='multiclass', num_classes=27).to(device)

In [ ]:
import glob

checkpoint_dir = os.path.join(BASE_PATH, 'models')
checkpoint_files = sorted(glob.glob(os.path.join(checkpoint_dir, 'checkpoint_*.pth')))

start_epoch = 1
if checkpoint_files:
    latest_checkpoint = checkpoint_files[-1]
    checkpoint = torch.load(latest_checkpoint, map_location=device)
    model.load_state_dict(checkpoint)
    print(f'Loaded checkpoint from {latest_checkpoint}')
    start_epoch = int(os.path.basename(latest_checkpoint).split('_')[1].split('.')[0]) + 1
else:
    print('No previous checkpoint found. Starting training from scratch.')


In [ ]:
model.train()
for epoch in range(start_epoch, EPOCHS + 1):
    running_loss = 0.0
    for image, crop, disease in train_loader:
        optimizer.zero_grad()
        image = image.to(device)
        crop = crop.to(device)
        disease = disease.to(device)

        plant_outputs, disease_outputs = model.forward(image)
        plant_loss = criterion(plant_outputs, crop)
        disease_loss = criterion(disease_outputs, disease)
        f1score_plant.update(plant_outputs, crop)
        accuracy_plant.update(plant_outputs, crop)

        f1score_disease.update(disease_outputs, disease)
        accuracy_disease.update(disease_outputs, disease)
        combined_loss = plant_loss + disease_loss
        combined_loss.backward()

        running_loss += combined_loss.item()
        optimizer.step()

    f1_plant = f1score_plant.compute().item()
    acc_plant = accuracy_plant.compute().item()
    f1_disease = f1score_disease.compute().item()
    acc_disease = accuracy_disease.compute().item()
    print(f'Epoch|{epoch} Loss| {running_loss/len(train_loader):.2f} | Plant F1 Score| {f1_plant*100:.2f}% | Disease F1 Score| {f1_disease*100:.2f}%  | Plant Accuracy| {acc_plant*100:.2f}% | Disease Accuracy| {acc_disease*100:.2f}%')

    f1score_plant.reset()
    f1score_disease.reset()
    accuracy_plant.reset()
    accuracy_disease.reset()

    if epoch % 5 == 0:
        torch.save(model.state_dict(), os.path.join(checkpoint_dir, f'checkpoint_{epoch}.pth'))
        print('Model check-point saved to models folder')

    model.eval()
    with torch.no_grad():
        for image, crop, disease in val_loader:
            image = image.to(device)
            crop = crop.to(device)
            disease = disease.to(device)

            plant_outputs, disease_outputs = model(image)
            f1score_plant.update(plant_outputs, crop)
            accuracy_plant.update(plant_outputs, crop)

            f1score_disease.update(disease_outputs, disease)
            accuracy_disease.update(disease_outputs, disease)

        acc_plant = accuracy_plant.compute().item()
        acc_disease = accuracy_disease.compute().item()
        f1_plant = f1score_plant.compute().item()
        f1_disease = f1score_disease.compute().item()
        print('------------- VALIDATION -------------')
        print(f'Plant F1 Score-- {f1_plant*100:.2f}% -- Disease Accuracy-- {f1_disease*100:.2f}% --  Plant Accuracy-- {acc_plant*100:.2f}% -- Disease Accuracy-- {acc_disease*100:.2f}%')
        f1score_plant.reset()
        f1score_disease.reset()
        accuracy_plant.reset()
        accuracy_disease.reset()

torch.save(model.state_dict(), os.path.join(BASE_PATH, 'multi_head_plant_disease.pth'))